# Phase 5k: Supervised baseline, retrieval in predicted emotion space

Reviewer item 4: the probes show that AV is easy to learn from the embeddings, so the obvious practical system is to
retrieve by distance between **predicted** emotion points rather than by raw embedding similarity. This notebook adds
that baseline under the primary (artist-filtered) protocol and compares it with the label-free strategies of Phase 5j.

**Training data.** Probes are fitted on the shared **train** split only (70/15/15: 1,187; 40/30/30: 475). Every
catalog track needs a prediction:
* train-split tracks: 5-fold cross-fitted, folds grouped by artist (GroupKFold), so no track's own label, or its
  artist's other labels, enters its prediction;
* all other catalog tracks (validation, test, and tracks outside the shared subsets): the model fitted on the full
  train split.

Test labels are never used. Choices (MuQ layer, classifier C, the late-fusion weight of the hybrid) are made on validation P@10.

**Variants.**
* `pred-AV (audio)`, `pred-AV (lyrics)`, `pred-AV (audio+lyrics)`: RidgeCV (logspace(−2, 5, 15), as in the coherence
  probes) on z-scored features → (arousal, valence); rank by Euclidean distance between predicted points.
* `pred-quadrant (audio+lyrics)`: multinomial logistic regression on the concatenated features → 4 quadrant
  probabilities; rank by Euclidean distance between probability vectors.
* `hybrid`: late fusion of label-free similarity (MuQ+SBERT late fusion) and negative predicted-AV distance, weight
  on validation; tests whether label-free similarity adds anything once a supervised signal exists.

**Sensitivity:** MERGE's splits are not artist-disjoint, so a test query's artist can appear in training. The
`artist-disjoint training` variant drops from training every track whose artist appears in that split's validation
or test subsets.

Outputs: `supervised_baseline_table.csv`, `supervised_baseline_tests.csv`, `supervised_baseline_selection.csv`,
`supervised_baseline_probe_r2.csv`.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, t as t_dist
from sklearn.preprocessing import StandardScaler

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR = os.path.join(ROOT, 'outputs')
ENC_DIR = os.path.join(OUT_DIR, 'encoders')
SIGMA, KMAX, N_BOOT, SEED = 0.10, 20, 10_000, 42
ALPHAS  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
SCHEMES = {'tvt_70_15_15': '70/15/15', 'tvt_40_30_30': '40/30/30'}
QUADS   = ['Q1', 'Q2', 'Q3', 'Q4']
QLAB    = {'Q1': 'Happy', 'Q2': 'Angry', 'Q3': 'Sad', 'Q4': 'Calm'}
rng = np.random.default_rng(SEED)

lyrics_df = pd.read_parquet(os.path.join(OUT_DIR, 'lyrics_features.parquet'))
merged    = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet'))
cat_sids  = lyrics_df['song_id'].tolist()
cat_pos   = {s: i for i, s in enumerate(cat_sids)}
a_ids = pd.read_csv(os.path.join(ENC_DIR, 'audio_song_ids.csv'))['song_id'].tolist()
a_row = [{s: i for i, s in enumerate(a_ids)}[s] for s in cat_sids]
assert pd.read_csv(os.path.join(ENC_DIR, 'text_song_ids.csv'))['song_id'].tolist() == cat_sids

muq  = np.load(os.path.join(ENC_DIR, 'audio_muq.npy'),  mmap_mode='r')[a_row].astype(np.float64)
mert = np.load(os.path.join(ENC_DIR, 'audio_mert.npy'), mmap_mode='r')[a_row].astype(np.float64)
a_old = pd.read_parquet(os.path.join(OUT_DIR, 'audio_features.parquet')).set_index('song_id').loc[cat_sids]
REPS = {'sbert': np.load(os.path.join(ENC_DIR, 'text_sbert_mpnet_trunc.npy')).astype(np.float64),
        'clap':  np.vstack(a_old['audio_clap'].values).astype(np.float64)}
for L in range(13):
    REPS[f'muq_L{L}'] = muq[:, L]; REPS[f'mert_L{L}'] = mert[:, L]
del muq, mert
FAMILY = lambda r: r.split('_L')[0]

lab  = merged.set_index('song_id').loc[cat_sids]
quad = lab['quadrant'].values
Y    = lab[['arousal', 'valence']].values.astype(np.float64)
D_AV = np.linalg.norm(Y[:, None] - Y[None], axis=-1)

mm = merged[merged['is_multimodal']]
lyr_stem_to_sid = {os.path.splitext(os.path.basename(p))[0]: s for s, p in zip(mm['song_id'], mm['lyrics_path'])}
def read_split(modality, scheme, subset):
    folder = 'MERGE_Audio_Balanced' if modality == 'audio' else 'MERGE_Lyrics_Balanced'
    ids = pd.read_csv(os.path.join(ROOT, folder, 'tvt_dataframes', scheme,
                                   f'{scheme}_{subset}_{modality}_balanced.csv'))['Song']
    return set(ids.map(lyr_stem_to_sid).dropna()) if modality == 'lyrics' else set(ids)
cat = set(cat_sids)
shared = {lbl: {sub: sorted(read_split('audio', sc, sub) & read_split('lyrics', sc, sub) & cat)
                for sub in ['train', 'validate', 'test']} for sc, lbl in SCHEMES.items()}
print({lbl: {k: len(v) for k, v in d.items()} for lbl, d in shared.items()})

{'70/15/15': {'train': 1187, 'validate': 305, 'test': 302}, '40/30/30': {'train': 475, 'validate': 616, 'test': 617}}


In [2]:
import re, unicodedata
def _norm_artist(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower().strip()
    return re.sub(r'[^a-z0-9]+', ' ', re.sub(r'^the\s+', '', s)).strip()
ART_ID = np.array([_norm_artist(a) for a in lab['artist'].values])
ART = ART_ID[:, None] == ART_ID[None]
np.fill_diagonal(ART, False)                          # the query itself is excluded separately
print(f'{len(set(ART_ID))} artist groups; mean same-artist candidates per track {ART.sum(1).mean():.2f}')

1086 artist groups; mean same-artist candidates per track 3.48


In [3]:
_unit = {}
def unit(rep, how):
    if (rep, how) not in _unit:
        X = REPS[rep]; X = StandardScaler().fit_transform(X) if how == 'z' else X
        _unit[(rep, how)] = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    return _unit[(rep, how)]

def score_matrix(cfg, rows):
    # cfg: ('single', (rep, how)) | ('late', (repA, howA), (repT, howT), alpha) | ('early', (repA, howA), (repT, howT))
    if cfg[0] == 'single':
        X = unit(*cfg[1]); return X[rows] @ X.T
    A, T = unit(*cfg[1]), unit(*cfg[2])
    if cfg[0] == 'late':
        return cfg[3] * (A[rows] @ A.T) + (1 - cfg[3]) * (T[rows] @ T.T)
    E = np.hstack([A, T]) / np.sqrt(2)                                     # unit-norm concatenation
    return E[rows] @ E.T

DISC = 1 / np.log2(np.arange(2, KMAX + 2))
def topk(S, k):
    # same selection as Phase 5e (argpartition at exactly k, then sort): duplicate tracks in MERGE give exact score
    # ties, and partitioning at a different k can break them differently, so each k is partitioned on its own
    top = np.argpartition(-S, k, 1)[:, :k]
    return np.take_along_axis(top, np.take_along_axis(S, top, 1).argsort(1)[:, ::-1], 1)

def evaluate(cfg, qs, exclude=None):
    # exclude: optional boolean (N, N) mask of catalog pairs that may not be retrieved for each other
    rows = np.array([cat_pos[s] for s in qs])
    S = score_matrix(cfg, rows); S[np.arange(len(rows)), rows] = -np.inf
    if exclude is not None:
        S[exclude[rows]] = -np.inf
    PK = np.column_stack([(quad[topk(S, k)] == quad[rows][:, None]).mean(1) for k in range(1, KMAX + 1)])
    top = topk(S, 10)
    d = np.take_along_axis(D_AV[rows], top, 1)
    Dq = D_AV[rows].copy(); Dq[np.arange(len(rows)), rows] = np.inf
    if exclude is not None:
        Dq[exclude[rows]] = np.inf
    g = np.exp(-d**2 / (2 * SIGMA**2)); gi = np.exp(-np.sort(Dq, 1)[:, :10]**2 / (2 * SIGMA**2))
    return {'P@K': PK, 'nDCG@10': (g * DISC[:10]).sum(1) / (gi * DISC[:10]).sum(1),
            'AVdist@10': d.mean(1), 'quadrant': quad[rows], 'song_id': np.array(qs)}

def best_single(lbl, family):
    reps = [r for r in REPS if FAMILY(r) == family]
    cands = [(r, h) for r in reps for h in ['raw', 'z']]
    return max(cands, key=lambda c: evaluate(('single', c), shared[lbl]['validate'], exclude=ART)['P@K'][:, 9].mean())

def best_alpha(lbl, ca, ct):
    return max(ALPHAS, key=lambda al: evaluate(('late', ca, ct, al), shared[lbl]['validate'], exclude=ART)['P@K'][:, 9].mean())

CFG = {}
for lbl in SCHEMES.values():
    c = {f: best_single(lbl, f) for f in ['muq', 'mert', 'sbert', 'clap']}
    a_muq, a_clap = best_alpha(lbl, c['muq'], c['sbert']), best_alpha(lbl, c['clap'], c['sbert'])
    CFG[lbl] = {
        'audio-only (MuQ)':         ('single', c['muq']),
        'lyrics-only (SBERT)':      ('single', c['sbert']),
        f'late fusion (α={a_muq})': ('late', c['muq'], c['sbert'], a_muq),
        'early fusion (≡ α=0.5)':   ('early', c['muq'], c['sbert']),
        'audio-only (CLAP)':        ('single', c['clap']),
        f'late fusion CLAP+SBERT (α={a_clap})': ('late', c['clap'], c['sbert'], a_clap),
        'audio-only (MERT)':        ('single', c['mert']),
    }
    print(lbl, {k: v[1:] for k, v in CFG[lbl].items()})

# early fusion == late fusion at alpha = 0.5 (unit-norm concatenation)
q = shared['70/15/15']['test'][:50]; rows = np.array([cat_pos[s] for s in q])
c = CFG['70/15/15']
assert np.allclose(score_matrix(('early', c['audio-only (MuQ)'][1], c['lyrics-only (SBERT)'][1]), rows),
                   score_matrix(('late', c['audio-only (MuQ)'][1], c['lyrics-only (SBERT)'][1], 0.5), rows))
print('early fusion == late fusion at α = 0.5: verified')

70/15/15 {'audio-only (MuQ)': (('muq_L6', 'z'),), 'lyrics-only (SBERT)': (('sbert', 'z'),), 'late fusion (α=0.6)': (('muq_L6', 'z'), ('sbert', 'z'), 0.6), 'early fusion (≡ α=0.5)': (('muq_L6', 'z'), ('sbert', 'z')), 'audio-only (CLAP)': (('clap', 'z'),), 'late fusion CLAP+SBERT (α=0.1)': (('clap', 'z'), ('sbert', 'z'), 0.1), 'audio-only (MERT)': (('mert_L7', 'z'),)}


40/30/30 {'audio-only (MuQ)': (('muq_L6', 'z'),), 'lyrics-only (SBERT)': (('sbert', 'z'),), 'late fusion (α=0.6)': (('muq_L6', 'z'), ('sbert', 'z'), 0.6), 'early fusion (≡ α=0.5)': (('muq_L6', 'z'), ('sbert', 'z')), 'audio-only (CLAP)': (('clap', 'z'),), 'late fusion CLAP+SBERT (α=0.3)': (('clap', 'z'), ('sbert', 'z'), 0.3), 'audio-only (MERT)': (('mert_L6', 'z'),)}
early fusion == late fusion at α = 0.5: verified


## Predictions for every catalog track

In [4]:
import warnings
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
warnings.filterwarnings('ignore')

RIDGE_ALPHAS = np.logspace(-2, 5, 15)
QIDX = {q: i for i, q in enumerate(QUADS)}
YQ = np.array([QIDX[q] for q in quad])
N = len(cat_sids)

def boot_ci(x, n=N_BOOT):
    idx = rng.integers(0, len(x), (n, len(x)))
    return np.percentile(x[idx].mean(1), [2.5, 97.5])

def holm(p):
    p = np.asarray(p); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def ridge():   return make_pipeline(StandardScaler(), RidgeCV(alphas=RIDGE_ALPHAS))
def logreg(C): return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))

def train_rows(lbl, artist_disjoint=False):
    tr = np.array([cat_pos[s] for s in shared[lbl]['train']])
    if artist_disjoint:
        held = {ART_ID[cat_pos[s]] for sub in ['validate', 'test'] for s in shared[lbl][sub]}
        tr = tr[~np.isin(ART_ID[tr], list(held))]
    return tr

def crossfit(X, target, make_model, tr, proba=False):
    """Cross-fitted (artist-grouped) predictions for train rows, full-train predictions for all other rows."""
    out = np.zeros((N, 4 if proba else target.shape[1]))
    pred = (lambda m, Z: m.predict_proba(Z)) if proba else (lambda m, Z: m.predict(Z))
    for fit_i, hold_i in GroupKFold(n_splits=5).split(tr, groups=ART_ID[tr]):
        m = make_model().fit(X[tr[fit_i]], target[tr[fit_i]]); out[tr[hold_i]] = pred(m, X[tr[hold_i]])
    rest = np.setdiff1d(np.arange(N), tr)
    m = make_model().fit(X[tr], target[tr]); out[rest] = pred(m, X[rest])
    return out

def neg_dist(P):
    return lambda rows: -np.linalg.norm(P[rows][:, None] - P[None], axis=-1)

def evaluate_S(score_fn, qs, exclude=ART):
    """Same metrics as evaluate(), for an arbitrary score function rows -> (len(rows), N)."""
    rows = np.array([cat_pos[s] for s in qs])
    S = score_fn(rows).astype(np.float64); S[np.arange(len(rows)), rows] = -np.inf
    if exclude is not None:
        S[exclude[rows]] = -np.inf
    PK = np.column_stack([(quad[topk(S, k)] == quad[rows][:, None]).mean(1) for k in range(1, KMAX + 1)])
    top = topk(S, 10); d = np.take_along_axis(D_AV[rows], top, 1)
    Dq = D_AV[rows].copy(); Dq[np.arange(len(rows)), rows] = np.inf
    if exclude is not None:
        Dq[exclude[rows]] = np.inf
    g = np.exp(-d**2 / (2 * SIGMA**2)); gi = np.exp(-np.sort(Dq, 1)[:, :10]**2 / (2 * SIGMA**2))
    return {'P@K': PK, 'nDCG@10': (g * DISC[:10]).sum(1) / (gi * DISC[:10]).sum(1), 'AVdist@10': d.mean(1),
            'quadrant': quad[rows], 'song_id': np.array(qs)}

# sanity: evaluate_S reproduces evaluate() for the label-free late fusion
lbl0 = '70/15/15'; late0 = [k for k in CFG[lbl0] if k.startswith('late fusion (')][0]
a = evaluate(CFG[lbl0][late0], shared[lbl0]['test'], exclude=ART)
b = evaluate_S(lambda r: score_matrix(CFG[lbl0][late0], r), shared[lbl0]['test'])
assert np.allclose(a['P@K'], b['P@K']) and np.allclose(a['nDCG@10'], b['nDCG@10'])
print('evaluate_S matches evaluate.')

evaluate_S matches evaluate.


In [5]:
val_p10 = lambda fn, lbl: evaluate_S(fn, shared[lbl]['validate'])['P@K'][:, 9].mean()
Zs = lambda X: StandardScaler().fit_transform(X)   # label-free scaling of each block before concatenation
LYR = REPS['sbert']

PRED, SEL, R2 = {}, [], []
for training in ['shared train', 'artist-disjoint training']:
    for lbl in SCHEMES.values():
        tr = train_rows(lbl, training != 'shared train')
        # audio layer chosen on validation P@10 of predicted-AV retrieval
        layer_scores = {}
        for L in range(13):
            P = crossfit(REPS[f'muq_L{L}'], Y, ridge, tr); layer_scores[L] = val_p10(neg_dist(P), lbl)
        L = max(layer_scores, key=layer_scores.get)
        XA = REPS[f'muq_L{L}']; XAL = np.hstack([Zs(XA), Zs(LYR)])
        P_a, P_l, P_al = crossfit(XA, Y, ridge, tr), crossfit(LYR, Y, ridge, tr), crossfit(XAL, Y, ridge, tr)
        Cs = {C: crossfit(XAL, YQ, lambda C=C: logreg(C), tr, proba=True) for C in [1e-5, 3e-5, 1e-4, 3e-4, 0.001, 0.003, 0.01, 0.1, 1.0]}
        C = max(Cs, key=lambda c: val_p10(neg_dist(Cs[c]), lbl))
        late = [k for k in CFG[lbl] if k.startswith('late fusion (')][0]
        S_late = lambda r, late=late, lbl=lbl: score_matrix(CFG[lbl][late], r)
        def hybrid(w, P=P_al, S_late=S_late):
            # both terms rescaled to comparable spread: similarity in [-1, 1], AV distance in [0, ~1.4]
            return lambda r: w * S_late(r) + (1 - w) * neg_dist(P)(r)
        w = max([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9], key=lambda w: val_p10(hybrid(w), lbl))
        PRED[(training, lbl)] = {'pred-AV (audio)': neg_dist(P_a), 'pred-AV (lyrics)': neg_dist(P_l),
                                 'pred-AV (audio+lyrics)': neg_dist(P_al), 'pred-quadrant (audio+lyrics)': neg_dist(Cs[C]),
                                 f'hybrid: late fusion + pred-AV (w={w})': hybrid(w)}
        SEL.append({'training': training, 'split': lbl, 'n_train': len(tr), 'muq_layer': L,
                    'val_P@10_by_layer': {k: round(v, 3) for k, v in layer_scores.items()}, 'logreg_C': C, 'hybrid_w': w})
        te = np.array([cat_pos[s] for s in shared[lbl]['test']])
        for name, P in [('audio', P_a), ('lyrics', P_l), ('audio+lyrics', P_al)]:
            R2.append({'training': training, 'split': lbl, 'features': name,
                       'R2_arousal': r2_score(Y[te, 0], P[te, 0]), 'R2_valence': r2_score(Y[te, 1], P[te, 1]),
                       'quadrant_acc_from_AV': np.mean(np.where(P[te, 0] >= .5, np.where(P[te, 1] >= .5, 'Q1', 'Q2'), np.where(P[te, 1] >= .5, 'Q4', 'Q3')) == quad[te])})
        R2.append({'training': training, 'split': lbl, 'features': f'audio+lyrics logistic (C={C})',
                   'quadrant_acc_from_AV': np.mean(np.array(QUADS)[Cs[C][te].argmax(1)] == quad[te])})
        print(training, lbl, f'n_train={len(tr)}  MuQ layer {L}  C={C}  hybrid w={w}')
sel = pd.DataFrame(SEL); sel.to_csv(os.path.join(OUT_DIR, 'supervised_baseline_selection.csv'), index=False)
r2 = pd.DataFrame(R2); r2.to_csv(os.path.join(OUT_DIR, 'supervised_baseline_probe_r2.csv'), index=False)
print(r2.round(3).to_string(index=False))

shared train 70/15/15 n_train=1187  MuQ layer 8  C=0.003  hybrid w=0.3


shared train 40/30/30 n_train=475  MuQ layer 9  C=0.001  hybrid w=0.4


artist-disjoint training 70/15/15 n_train=705  MuQ layer 8  C=0.0003  hybrid w=0.4


artist-disjoint training 40/30/30 n_train=248  MuQ layer 9  C=0.003  hybrid w=0.6
                training    split                         features  R2_arousal  R2_valence  quadrant_acc_from_AV
            shared train 70/15/15                            audio       0.516       0.404                 0.728
            shared train 70/15/15                           lyrics       0.373       0.519                 0.679
            shared train 70/15/15                     audio+lyrics       0.571       0.588                 0.791
            shared train 70/15/15  audio+lyrics logistic (C=0.003)         NaN         NaN                 0.848
            shared train 40/30/30                            audio       0.533       0.385                 0.681
            shared train 40/30/30                           lyrics       0.372       0.477                 0.658
            shared train 40/30/30                     audio+lyrics       0.572       0.558                 0.775
            sh

## Test results (artist-filtered protocol), with the label-free strategies for reference

In [6]:
RES = {}
for (training, lbl), fns in PRED.items():
    for name, fn in fns.items():
        RES[(training, lbl, name)] = evaluate_S(fn, shared[lbl]['test'])
for lbl in SCHEMES.values():
    for name in ['audio-only (MuQ)', 'lyrics-only (SBERT)', [k for k in CFG[lbl] if k.startswith('late fusion (')][0]]:
        RES[('label-free', lbl, name)] = evaluate(CFG[lbl][name], shared[lbl]['test'], exclude=ART)

ref = pd.read_csv(os.path.join(OUT_DIR, 'main_results_table_artist.csv'))
for lbl in SCHEMES.values():
    r = ref[(ref.split == lbl) & (ref.strategy == 'audio-only (MuQ)')].iloc[0]
    assert abs(r['P@10'] - RES[('label-free', lbl, 'audio-only (MuQ)')]['P@K'][:, 9].mean()) < 1e-9
print('Label-free rows match main_results_table_artist.csv.')

rows = []
for (training, lbl, name), m in RES.items():
    r = {'training': training, 'split': lbl, 'strategy': name, 'n': len(m['nDCG@10'])}
    for col, x in [('P@10', m['P@K'][:, 9]), ('nDCG@10', m['nDCG@10']), ('AVdist@10', m['AVdist@10'])]:
        lo, hi = boot_ci(x); r.update({col: x.mean(), f'{col}_lo': lo, f'{col}_hi': hi})
    rows.append(r)
table = pd.DataFrame(rows); table.to_csv(os.path.join(OUT_DIR, 'supervised_baseline_table.csv'), index=False)
for lbl in SCHEMES.values():
    print(f'\n=== {lbl} test, same artist excluded ===')
    t = table[table.split == lbl]
    print(t[['training', 'strategy', 'P@10', 'P@10_lo', 'P@10_hi', 'nDCG@10', 'AVdist@10']].round(3).to_string(index=False))

Label-free rows match main_results_table_artist.csv.



=== 70/15/15 test, same artist excluded ===
                training                              strategy  P@10  P@10_lo  P@10_hi  nDCG@10  AVdist@10
            shared train                       pred-AV (audio) 0.648    0.615    0.682    0.420      0.200
            shared train                      pred-AV (lyrics) 0.602    0.567    0.638    0.404      0.199
            shared train                pred-AV (audio+lyrics) 0.747    0.713    0.780    0.486      0.164
            shared train          pred-quadrant (audio+lyrics) 0.769    0.736    0.802    0.486      0.164
            shared train hybrid: late fusion + pred-AV (w=0.3) 0.781    0.749    0.812    0.520      0.154
artist-disjoint training                       pred-AV (audio) 0.622    0.587    0.655    0.409      0.209
artist-disjoint training                      pred-AV (lyrics) 0.571    0.536    0.605    0.386      0.210
artist-disjoint training                pred-AV (audio+lyrics) 0.710    0.677    0.744    0.464    

In [7]:
trows = []
for lbl in SCHEMES.values():
    late = [k for k in CFG[lbl] if k.startswith('late fusion (')][0]
    for training in ['shared train', 'artist-disjoint training']:
        hyb = [k for k in PRED[(training, lbl)] if k.startswith('hybrid')][0]
        pairs = [(('', 'pred-AV (audio+lyrics)'), ('label-free', late)),
                 (('', 'pred-quadrant (audio+lyrics)'), ('label-free', late)),
                 (('', 'pred-AV (audio)'), ('label-free', 'audio-only (MuQ)')),
                 (('', 'pred-AV (lyrics)'), ('label-free', 'lyrics-only (SBERT)')),
                 (('', hyb), ('', 'pred-AV (audio+lyrics)'))]
        for metric, get in [('P@10', lambda m: m['P@K'][:, 9]), ('nDCG@10', lambda m: m['nDCG@10'])]:
            block = []
            for (ta, a), (tb, b) in pairs:
                xa, xb = get(RES[(ta or training, lbl, a)]), get(RES[(tb or training, lbl, b)]); d = xa - xb; lo, hi = boot_ci(d)
                p = wilcoxon(xa, xb, zero_method='pratt').pvalue if (d != 0).any() else 1.0
                block.append({'training': training, 'split': lbl, 'metric': metric, 'comparison': f'{a} − {b}',
                              'diff': d.mean(), 'lo': lo, 'hi': hi, 'p': p})
            for r, ph in zip(block, holm([r['p'] for r in block])):
                r['p_holm'] = ph
            trows += block
tests = pd.DataFrame(trows)
tests.to_csv(os.path.join(OUT_DIR, 'supervised_baseline_tests.csv'), index=False)
print(tests.to_string(index=False, float_format=lambda x: f'{x:.3g}'))
print('Saved: supervised_baseline_table.csv, supervised_baseline_tests.csv, supervised_baseline_selection.csv, '
      'supervised_baseline_probe_r2.csv')

                training    split  metric                                                     comparison     diff       lo       hi        p   p_holm
            shared train 70/15/15    P@10                   pred-AV (audio+lyrics) − late fusion (α=0.6)   0.0573   0.0285   0.0868 5.83e-05 0.000233
            shared train 70/15/15    P@10             pred-quadrant (audio+lyrics) − late fusion (α=0.6)   0.0795   0.0503    0.108 1.49e-09 7.43e-09
            shared train 70/15/15    P@10                             pred-AV (audio) − audio-only (MuQ)   0.0384   0.0109   0.0659  0.00221  0.00442
            shared train 70/15/15    P@10                         pred-AV (lyrics) − lyrics-only (SBERT) 0.000331  -0.0285   0.0291    0.718    0.718
            shared train 70/15/15    P@10 hybrid: late fusion + pred-AV (w=0.3) − pred-AV (audio+lyrics)   0.0341   0.0169    0.052 0.000619  0.00186
            shared train 70/15/15 nDCG@10                   pred-AV (audio+lyrics) − late fusion (α=